Dependencias básicas

In [ ]:
%pip -q install pydicom scikit-image scikit-learn tqdm
# TensorFlow ya viene en Colab. Si te tira error de compatibilidad, prueba:
# %pip -q install "tensorflow>=2.15,<3.0"


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 19.3 MB/s eta 0:00:00


Montaje de Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

from pathlib import Path

BASE_PATH  = Path("/content/drive/MyDrive/lidc/LIDC-IDRI")
DICOM_ROOT = BASE_PATH

print("BASE_PATH:", BASE_PATH.exists(), BASE_PATH)
print("DICOM_ROOT:", DICOM_ROOT.exists(), DICOM_ROOT)


Mounted at /content/drive
BASE_PATH: True /content/drive/MyDrive/lidc/LIDC-IDRI
DICOM_ROOT: True /content/drive/MyDrive/lidc/LIDC-IDRI


Fija rutas

In [ ]:
# === Rutas en Drive y reparación de índices ===
from pathlib import Path
import pandas as pd

BASE_PATH = Path("/content/drive/MyDrive/lidc")
DICOM_ROOT = BASE_PATH / "LIDC-IDRI"

print("BASE_PATH:", BASE_PATH.exists(), BASE_PATH)
print("DICOM_ROOT:", DICOM_ROOT.exists(), DICOM_ROOT)

# Repara rutas antiguas en dataset_index/resolved_index si existen
for name in ["dataset_index.csv", "resolved_index.csv"]:
    p = BASE_PATH / name
    if p.exists():
        df = pd.read_csv(p)
        for col in ["example_dicom_path", "series_dir"]:
            if col in df.columns:
                df[col] = df[col].astype(str).str.replace(
                    "/content/lidc_remote", str(BASE_PATH), regex=False
                )
        df.to_csv(p, index=False)
        print(f"✔️ Reparado {name}")
    else:
        print(f"(aviso) No existe {name}, se creará si es necesario.")


BASE_PATH: True /content/drive/MyDrive/lidc
DICOM_ROOT: True /content/drive/MyDrive/lidc/LIDC-IDRI
✔️ Reparado dataset_index.csv
✔️ Reparado resolved_index.csv


DataPrep

In [ ]:
import re, pandas as pd
from pathlib import Path

# --- RUTAS EN DRIVE ---
ROOT = Path("/content/drive/MyDrive/lidc")     # aquí están los Excels
DICOM_ROOT = ROOT / "LIDC-IDRI"                # aquí están los DICOMs

# Busca Excels en la raíz (o subcarpetas) de lidc
XLS_COUNTS = next(ROOT.glob("**/*nodule-counts*.xls*"), None)
XLS_DIAG   = next(ROOT.glob("**/*diagnosis*.xls*"), None)

print("ROOT:", ROOT.exists(), ROOT)
print("DICOM_ROOT:", DICOM_ROOT.exists(), DICOM_ROOT)
print("XLS_COUNTS:", XLS_COUNTS)
print("XLS_DIAG:", XLS_DIAG)

def clean_cols(df):
    df = df.copy()
    df.columns = (df.columns
                  .str.strip()
                  .str.replace(r"\s+", " ", regex=True)
                  .str.replace("\n"," ", regex=False)
                  .str.lower())
    return df

def read_excel_robust(path: Path):
    ext = path.suffix.lower()
    if ext == ".xls":
        return pd.read_excel(path, engine="xlrd")
    elif ext in (".xlsx",".xlsm",".xltx",".xltm"):
        return pd.read_excel(path, engine="openpyxl")
    return pd.read_excel(path)

def find_col(cols, *patterns):
    for pat in patterns:
        for c in cols:
            if re.search(pat, c):
                return c
    return None

DATASET_CSV = ROOT / "dataset_index.csv"

if DATASET_CSV.exists():
    print("🔹 Usando dataset_index.csv ya existente:", DATASET_CSV)
    df_ready = pd.read_csv(DATASET_CSV)
else:
    assert DICOM_ROOT.exists(), "No se encontró la carpeta LIDC-IDRI en Drive"
    assert XLS_COUNTS is not None, "Falta Excel de nódulos"
    assert XLS_DIAG   is not None, "Falta Excel de diagnóstico"

    df_counts = clean_cols(read_excel_robust(XLS_COUNTS))
    df_diag   = clean_cols(read_excel_robust(XLS_DIAG))

    c_id  = find_col(df_counts.columns, r'^tcia patient id$','^patient id$','^tcia.*id$')
    c_tot = find_col(df_counts.columns, r'^total number of nodules')
    c_ge3 = find_col(df_counts.columns, r'number of nodules *>=? *3mm')
    c_lt3 = find_col(df_counts.columns, r'number of nodules *< *3mm')

    df_counts_std = df_counts.rename(columns={
        c_id:  "patient_id",
        c_tot: "nodules_total",
        c_ge3: "nodules_ge_3mm",
        c_lt3: "nodules_lt_3mm"
    })[["patient_id","nodules_total","nodules_ge_3mm","nodules_lt_3mm"]]

    d_id   = find_col(df_diag.columns, r'^tcia patient id$','^patient id$')
    d_lvl  = find_col(df_diag.columns, r'^diagnosis at the patient level')
    d_meth = find_col(df_diag.columns, r'^diagnosis method')
    d_prim = find_col(df_diag.columns, r'^primary tumor site')

    df_diag_std = df_diag.rename(columns={
        d_id:   "patient_id",
        d_lvl:  "diagnosis_level",
        d_meth: "diagnosis_method",
        d_prim: "primary_tumor_site"
    })[["patient_id","diagnosis_level","diagnosis_method","primary_tumor_site"]]

    def map_label(x):
        if pd.isna(x): return None
        x = int(x)
        if x in (2,3): return 1
        if x in (0,1): return 0
        return None

    df_diag_std["label"] = df_diag_std["diagnosis_level"].apply(map_label)

    def normalize_pid(x):
        if pd.isna(x): return None
        s = str(x).strip().replace("_","-")
        m = re.search(r'(\d+)$', s)
        if m: return f"LIDC-IDRI-{int(m.group(1)):04d}"
        return s

    df_counts_std["patient_id"] = df_counts_std["patient_id"].apply(normalize_pid)
    df_diag_std["patient_id"]   = df_diag_std["patient_id"].apply(normalize_pid)

    # Indexar una ruta de ejemplo por paciente y conteo de DICOMs
    def first_dicom_path(patient_id):
        pdir = DICOM_ROOT / str(patient_id)
        if not pdir.exists():
            return None
        for fp in pdir.rglob("*.dcm"):
            return str(fp)
        return None

    df_merged = (df_counts_std
                 .merge(df_diag_std, on="patient_id", how="left")
                 .drop_duplicates(subset=["patient_id"])
                 .reset_index(drop=True))

    df_merged["example_dicom_path"] = df_merged["patient_id"].apply(first_dicom_path)
    df_merged["dicom_count"] = df_merged["patient_id"].apply(
        lambda pid: sum(1 for _ in (DICOM_ROOT/str(pid)).rglob("*.dcm"))
        if isinstance(pid, str) and (DICOM_ROOT/str(pid)).exists() else 0
    )

    df_ready = df_merged.query("dicom_count > 0 and label == label").reset_index(drop=True)
    df_ready.to_csv(DATASET_CSV, index=False)
    print("✅ Guardado:", DATASET_CSV)

print("Pacientes listos:", len(df_ready))
df_ready.head(3)


ROOT: True /content/drive/MyDrive/lidc
DICOM_ROOT: True /content/drive/MyDrive/lidc/LIDC-IDRI
XLS_COUNTS: /content/drive/MyDrive/lidc/lidc-idri-nodule-counts-6-23-2015.xlsx
XLS_DIAG: /content/drive/MyDrive/lidc/tcia-diagnosis-data-2012-04-20.xls
🔹 Usando dataset_index.csv ya existente: /content/drive/MyDrive/lidc/dataset_index.csv
Pacientes listos: 157


,patient_id,nodules_total,nodules_ge_3mm,nodules_lt_3mm,diagnosis_level,diagnosis_method,primary_tumor_site,label,dicom_count,example_dicom_path
0,LIDC-IDRI-0068,7,6,1,3.0,4.0,Head & Neck Cancer,1.0,262,/content/drive/MyDrive/lidc/LIDC-IDRI/LIDC-IDR...
1,LIDC-IDRI-0071,4,0,4,3.0,1.0,Head & Neck,1.0,262,/content/drive/MyDrive/lidc/LIDC-IDRI/LIDC-IDR...
2,LIDC-IDRI-0072,3,1,2,2.0,4.0,Lung Cancer,1.0,306,/content/drive/MyDrive/lidc/LIDC-IDRI/LIDC-IDR...


Resolver Series CT

In [ ]:
import pydicom
import pandas as pd
from pathlib import Path

def is_ct(fp):
    try:
        ds = pydicom.dcmread(fp, stop_before_pixels=True)
        return getattr(ds, "Modality", "") == "CT"
    except Exception:
        return False

rows = []
for pdir in sorted(DICOM_ROOT.glob("LIDC-IDRI-*")):
    if not pdir.is_dir():
        continue
    found = None
    for study in pdir.iterdir():
        if not study.is_dir():
            continue
        for series in study.iterdir():
            if not series.is_dir():
                continue
            first = next(series.glob("*.dcm"), None)
            if first and is_ct(first):
                num = sum(1 for _ in series.glob("*.dcm"))
                found = {"patient_id": pdir.name, "series_dir": str(series), "num_slices": num}
                break
        if found:
            break
    if found:
        rows.append(found)

res_path = BASE_PATH / "resolved_index.csv"
pd.DataFrame(rows).to_csv(res_path, index=False)
print("CT series resueltas:", len(rows), "->", res_path)


CT series resueltas: 1010 -> /content/drive/MyDrive/lidc/resolved_index.csv


Loader

In [ ]:
import pydicom, numpy as np
from pathlib import Path
from skimage.transform import resize

IMG_SIZE = 224

def _read_header(fp):
    return pydicom.dcmread(fp, stop_before_pixels=True)

def _is_ct(fp):
    try:
        return getattr(_read_header(fp), "Modality", "") == "CT"
    except:
        return False

def fast_series_from_example(example_dcm_path: str, max_files=512):
    """Dentro del mismo estudio, busca la serie CT con más archivos y los ordena."""
    ex = Path(example_dcm_path)
    series_dir = ex.parent
    study_dir  = series_dir.parent

    best_files = []
    for ser_dir in study_dir.iterdir():
        if not ser_dir.is_dir(): continue
        files = list(ser_dir.glob("*.dcm"))
        if not files: continue
        if not any(_is_ct(fp) for fp in files[:2]): continue
        if len(files) > len(best_files): best_files = files

    if not best_files:
        raise RuntimeError("No se encontró serie CT en este estudio.")

    # forma (Rows, Cols) dominante
    from collections import Counter
    shapes = []
    for fp in best_files[:max_files]:
        try:
            ds = _read_header(fp)
            shapes.append((int(getattr(ds,"Rows",0)), int(getattr(ds,"Columns",0))))
        except:
            pass
    (R,C), _ = Counter(shapes).most_common(1)[0]
    files_ok = [fp for fp in best_files if getattr(_read_header(fp),"Rows",None)==R and getattr(_read_header(fp),"Columns",None)==C][:max_files]

    def sort_key(fp):
        ds = _read_header(fp)
        if hasattr(ds, "ImagePositionPatient"):
            return float(ds.ImagePositionPatient[2])
        return int(getattr(ds, "InstanceNumber", 0))
    files_sorted = sorted(files_ok, key=sort_key)
    return [pydicom.dcmread(fp) for fp in files_sorted]

def to_hu(ds):
    arr = ds.pixel_array.astype(np.int16)
    slope = getattr(ds, "RescaleSlope", 1)
    intercept = getattr(ds, "RescaleIntercept", 0)
    return arr * slope + intercept

def window_lung(img_hu, center=-600, width=1500):
    low, high = center - width/2, center + width/2
    img = np.clip(img_hu, low, high)
    return (img - low) / (high - low + 1e-6)

def load_central_slice_normalized(example_dcm_path: str):
    dsets = fast_series_from_example(example_dcm_path)
    stack = np.stack([to_hu(ds) for ds in dsets], axis=0)
    mid = stack.shape[0] // 2
    img = window_lung(stack[mid])
    img = resize(img, (IMG_SIZE, IMG_SIZE), preserve_range=True, anti_aliasing=True).astype(np.float32)
    return np.expand_dims(img, axis=-1)


Pre-Checkeo


In [ ]:
from pathlib import Path
import pandas as pd

ROOT = Path("/content/drive/MyDrive/lidc")     # carpeta raíz en Drive
DICOM_ROOT = ROOT / "LIDC-IDRI"

# Debe existir el resolved_index.csv (si no, vuelve a correr “Resolver Series CT”)
RES_PATH = ROOT / "resolved_index.csv"
IDX_PATH = ROOT / "dataset_index.csv"
assert RES_PATH.exists(), "No existe resolved_index.csv. Vuelve a correr el bloque 'Resolver Series CT'."
assert IDX_PATH.exists(), "No existe dataset_index.csv. Vuelve a correr DataPrep."

res = pd.read_csv(RES_PATH)               # contiene patient_id, series_dir, num_slices
idx = pd.read_csv(IDX_PATH)               # contiene patient_id, label (y otros)

# Merge para quedarnos con label + series_dir
df = (idx.merge(res, on="patient_id", how="inner")
         .dropna(subset=["series_dir","label"])
         .reset_index(drop=True))

# Filtrar series que realmente existen y tienen dcm
from pathlib import Path
def has_dicoms(ser_dir):
    p = Path(str(ser_dir))
    return p.exists() and any(p.glob("*.dcm"))
df = df[df["series_dir"].apply(has_dicoms)].reset_index(drop=True)

print("Series CT válidas encontradas:", len(df))
print(df[["patient_id","label","series_dir","num_slices"]].head(5))


Series CT válidas encontradas: 157
       patient_id  label                                         series_dir  \
0  LIDC-IDRI-0068    1.0  /content/drive/MyDrive/lidc/LIDC-IDRI/LIDC-IDR...   
1  LIDC-IDRI-0071    1.0  /content/drive/MyDrive/lidc/LIDC-IDRI/LIDC-IDR...   
2  LIDC-IDRI-0072    1.0  /content/drive/MyDrive/lidc/LIDC-IDRI/LIDC-IDR...   
3  LIDC-IDRI-0088    1.0  /content/drive/MyDrive/lidc/LIDC-IDRI/LIDC-IDR...   
4  LIDC-IDRI-0090    1.0  /content/drive/MyDrive/lidc/LIDC-IDRI/LIDC-IDR...   

   num_slices  
0         261  
1         261  
2         305  
3         241  
4         133  


Split y tf.data por batch

In [ ]:
import numpy as np, tensorflow as tf, math, pydicom
from skimage.transform import resize
from pathlib import Path

IMG_SIZE = 224
AUTOTUNE = tf.data.AUTOTUNE
BATCH = 8
EPOCHS = 8

# ---------- Utilidades de carga, ahora basadas en series_dir ----------
def _read_hdr(fp):
    return pydicom.dcmread(fp, stop_before_pixels=True)

def _sort_key(fp):
    ds = _read_hdr(fp)
    if hasattr(ds, "ImagePositionPatient"):
        try:
            return float(ds.ImagePositionPatient[2])
        except:
            pass
    return int(getattr(ds, "InstanceNumber", 0))

def _to_hu(ds):
    arr = ds.pixel_array.astype(np.int16)
    slope = getattr(ds, "RescaleSlope", 1)
    inter = getattr(ds, "RescaleIntercept", 0)
    return arr * slope + inter

def _window_lung(img_hu, center=-600, width=1500):
    lo, hi = center - width/2, center + width/2
    img = np.clip(img_hu, lo, hi)
    return (img - lo) / (hi - lo + 1e-6)

def load_central_from_series(series_dir: str):
    sdir = Path(series_dir)
    files = sorted(list(sdir.glob("*.dcm")), key=_sort_key)
    if not files:
        raise RuntimeError(f"Sin DICOMs en {series_dir}")
    # lee una sola vez a HU y extrae la rebanada central
    stacks = []
    for fp in files:
        try:
            ds = pydicom.dcmread(fp)
            stacks.append(_to_hu(ds))
        except:
            pass
    if not stacks:
        raise RuntimeError(f"No se pudieron leer DICOMs en {series_dir}")
    stack = np.stack(stacks, axis=0)
    mid = stack.shape[0] // 2
    img = _window_lung(stack[mid])
    img = resize(img, (IMG_SIZE, IMG_SIZE), preserve_range=True, anti_aliasing=True).astype(np.float32)
    return np.expand_dims(img, -1)  # (H,W,1)

# ---------- Armar listas (series_dir, label) ----------
records = df[["series_dir","label"]].copy()
records["series_dir"] = records["series_dir"].astype(str)
records["label"] = records["label"].astype(int)

# Split estratificado simple
from sklearn.model_selection import train_test_split
train_df, val_df = train_test_split(
    records, test_size=0.20, random_state=42, stratify=records["label"]
)

print(f"train: {len(train_df)}   val: {len(val_df)}")
print("balance train:", dict(train_df["label"].value_counts()))
print("balance   val:", dict(val_df["label"].value_counts()))

train_records = list(train_df.itertuples(index=False, name=None))  # [(series_dir, label), ...]
val_records   = list(val_df.itertuples(index=False, name=None))

# ---------- tf.data ----------
def gen(items):
    for ser, y in items:
        try:
            x = load_central_from_series(ser)
            yield x.astype(np.float32), np.int32(y)
        except Exception as e:
            print(f"[skip] {ser} -> {e}")

output_sig = (
    tf.TensorSpec(shape=(IMG_SIZE, IMG_SIZE, 1), dtype=tf.float32),
    tf.TensorSpec(shape=(), dtype=tf.int32),
)

ds_train = tf.data.Dataset.from_generator(lambda: gen(train_records), output_signature=output_sig)
ds_val   = tf.data.Dataset.from_generator(lambda: gen(val_records),   output_signature=output_sig)

def prep(ds, shuffle=False):
    if shuffle:
        ds = ds.shuffle(512, reshuffle_each_iteration=True)
    return (ds.batch(BATCH)
             .prefetch(AUTOTUNE))

ds_train = prep(ds_train, shuffle=True)
ds_val   = prep(ds_val,   shuffle=False)

# Pasos por época (evita "ran out of data")
steps_per_epoch  = math.ceil(len(train_records) / BATCH)
val_steps        = math.ceil(len(val_records)   / BATCH)
print(f"steps_per_epoch={steps_per_epoch}  val_steps={val_steps}")

# Sanity peek
sample_ok = next(iter(ds_train.take(1)), None)
if sample_ok is None:
    raise RuntimeError("ds_train vacío incluso tras generación; revisa rutas en Drive.")


train: 125   val: 32
balance train: {1: np.int64(75), 0: np.int64(50)}
balance   val: {1: np.int64(19), 0: np.int64(13)}
steps_per_epoch=16  val_steps=4


Check Sanity

In [ ]:
import tensorflow as tf

# OJO: la cardinalidad -2 = UNKNOWN; no la uses para decidir vacío.
# En su lugar, intenta tomar 1 batch.
try:
    _sample = next(iter(ds_train.take(1)))
    print("✅ ds_train entrega batches OK.")
except StopIteration:
    raise RuntimeError("ds_train está vacío: revisa rutas/filtros.")
except Exception as e:
    raise RuntimeError(f"Fallo cargando un batch de ds_train: {e}")


✅ ds_train entrega batches OK.


Modelo baseline y entrenamiento

In [ ]:
import math, numpy as np, tensorflow as tf
from tensorflow.keras import layers, models, callbacks

IMG_SIZE = 224
EPOCHS   = 8
BATCH    = 8  # mantén el mismo batch que usaste arriba

# Reusa pasos ya calculados arriba
# steps_per_epoch  = math.ceil(len(train_records) / BATCH)
# val_steps        = math.ceil(len(val_records)   / BATCH)

# --- Modelo baseline ---
model = models.Sequential([
    layers.Input((IMG_SIZE, IMG_SIZE, 1)),
    layers.Conv2D(16, 3, padding="same", activation="relu"),
    layers.MaxPool2D(),
    layers.Conv2D(32, 3, padding="same", activation="relu"),
    layers.MaxPool2D(),
    layers.Conv2D(64, 3, padding="same", activation="relu"),
    layers.GlobalAveragePooling2D(),
    layers.Dense(64, activation="relu"),
    layers.Dense(1, activation="sigmoid"),
])

model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy", tf.keras.metrics.AUC(name="auc")]
)

# --- class_weight desde el split real (train_df) ---
counts = train_df["label"].value_counts()
pos = int(counts.get(1, 0))
neg = int(counts.get(0, 0))
total = pos + neg if (pos + neg) > 0 else 1
class_weight = {
    0: total / (2 * max(neg, 1)),
    1: total / (2 * max(pos, 1)),
}
print("class_weight:", class_weight)

# --- Callbacks básicos (opcionales) ---
ckpt_path = (BASE_PATH / "modelo_baseline_colab.h5")
cbs = [
    callbacks.EarlyStopping(monitor="val_auc", mode="max", patience=3, restore_best_weights=True),
    callbacks.ModelCheckpoint(filepath=str(ckpt_path), monitor="val_auc", mode="max", save_best_only=True),
]

# --- Entrenamiento ---
history = model.fit(
    ds_train,
    validation_data=ds_val,
    epochs=EPOCHS,
    steps_per_epoch=steps_per_epoch,   # <- ya calculado en tu split
    validation_steps=val_steps,        # <- ya calculado en tu split
    class_weight=class_weight,
    callbacks=cbs,
    verbose=1
)

print("✅ Entrenamiento terminado. Modelo (mejor) guardado en:", ckpt_path)


class_weight: {0: 1.25, 1: 0.8333333333333334}
Epoch 1/8
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 388ms/step - accuracy: 0.3742 - auc: 0.3773 - loss: 0.6966

16/16 ━━━━━━━━━━━━━━━━━━━━ 2062s 106s/step - accuracy: 0.3752 - auc: 0.3796 - loss: 0.6965 - val_accuracy: 0.4062 - val_auc: 0.5547 - val_loss: 0.6962
Epoch 2/8


/usr/local/lib/python3.12/dist-packages/keras/src/trainers/epoch_iterator.py:160: UserWarning: Your input ran out of data; interrupting training. Make sure that your dataset or generator can generate at least `steps_per_epoch * epochs` batches. You may need to use the `.repeat()` function when building your dataset.
  self._interrupted_warning()


16/16 ━━━━━━━━━━━━━━━━━━━━ 114s 7s/step - accuracy: 0.0000e+00 - auc: 0.0000e+00 - loss: 0.0000e+00 - val_accuracy: 0.4062 - val_auc: 0.5547 - val_loss: 0.6962
Epoch 3/8
16/16 ━━━━━━━━━━━━━━━━━━━━ 5216s 56s/step - accuracy: 0.4416 - auc: 0.5311 - loss: 0.7041 - val_accuracy: 0.4375 - val_auc: 0.5000 - val_loss: 0.6932
Epoch 4/8
16/16 ━━━━━━━━━━━━━━━━━━━━ 116s 7s/step - accuracy: 0.0000e+00 - auc: 0.0000e+00 - loss: 0.0000e+00 - val_accuracy: 0.4375 - val_auc: 0.5000 - val_loss: 0.6932
✅ Entrenamiento terminado. Modelo (mejor) guardado en: /content/drive/MyDrive/lidc/modelo_baseline_colab.h5


Métricas y umbral

In [ ]:
import numpy as np
from sklearn.metrics import classification_report, confusion_matrix, roc_curve, auc as sk_auc

# Predicciones val
y_true, y_prob = [], []
for bx, by in ds_val:
    y_true.append(by.numpy())
    y_prob.append(model.predict(bx, verbose=0).ravel())
y_true = np.concatenate(y_true)
y_prob = np.concatenate(y_prob)

# 0.5 por defecto
y_pred = (y_prob >= 0.5).astype(int)

print(classification_report(y_true, y_pred, target_names=["benigno/unknown","maligno"], zero_division=0))
print("Confusion:\n", confusion_matrix(y_true, y_pred))

# ROC
fpr, tpr, th = roc_curve(y_true, y_prob)
print("ROC AUC:", sk_auc(fpr, tpr))

# Buscar mejor umbral por F1 de la clase positiva (maligno)
best_thr, best_f1 = 0.5, -1
for thr in np.linspace(0.1, 0.9, 17):
    pred = (y_prob >= thr).astype(int)
    tp = np.sum((y_true==1)&(pred==1))
    fp = np.sum((y_true==0)&(pred==1))
    fn = np.sum((y_true==1)&(pred==0))
    prec = tp/(tp+fp+1e-9)
    rec  = tp/(tp+fn+1e-9)
    f1   = 2*prec*rec/(prec+rec+1e-9)
    if f1 > best_f1:
        best_f1, best_thr = f1, thr
print(f"Mejor umbral F1(maligno): {best_thr:.2f} (F1={best_f1:.3f})")


                 precision    recall  f1-score   support

benigno/unknown       0.41      1.00      0.58        13
        maligno       0.00      0.00      0.00        19

       accuracy                           0.41        32
      macro avg       0.20      0.50      0.29        32
   weighted avg       0.17      0.41      0.23        32

Confusion:
 [[13  0]
 [19  0]]
ROC AUC: 0.5506072874493928
Mejor umbral F1(maligno): 0.10 (F1=0.745)


Guardado en Drive

In [ ]:
OUT_MODEL = BASE_PATH / "modelo_baseline_colab.h5"
model.save(OUT_MODEL)
print("Modelo guardado en:", OUT_MODEL)


Modelo guardado en: /content/drive/MyDrive/lidc/modelo_baseline_colab.h5
